# DS · 06 Forecast ARIMA



## 🎯 Objetivos

Al final de este notebook:
- [ ] Entenderás componentes de series de tiempo (tendencia, estacionalidad, ruido)
- [ ] Aplicarás ARIMA para forecast de demanda
- [ ] Evaluarás precisión con MAE, MAPE, RMSE
- [ ] Generarás pronósticos con intervalos de confianza

## 📊 Contexto de Supply Chain

**Problema de negocio**: Planear producción y compras sin saber demanda futura genera exceso/faltante de inventario.

**Impacto esperado**: Reducción 15-20% en stock outs, mejora 10-15% en inventory turns, menor obsolescencia.

**Cuando usar**: Ciclos de S&OP mensual, planeación de capacidad trim

estral, negociaciones con proveedores.

## Contexto de Negocio

Demanda histórica muestra patrones predecibles (trend + seasonality). Necesidad de forecast automático para planificación integrada.

## Qué / Por qué / Para qué / Cuándo / Cómo
- **Qué**: Modelado de forecast ARIMA/SARIMA: estimar demanda futura con intervalos de confianza.
- **Por qué**: Automatizar planning, reducir bias humano, cuantificar incertidumbre (95% CI) para decisiones robustas.
- **Para qué**: Input a S&OP, cálculo de stock de seguridad, presupuestos de venta, planificación de producción.
- **Cuándo**: Ejecución semanal/mensual; re-entrenar trimestralmente con nuevos datos.
- **Cómo**: Pruebas ADF/KPSS, grid search de parámetros (p,d,q), validación walk-forward, RMSE/MAPE como métricas.

## Contexto de Negocio

Demanda histórica muestra patrones predecibles (trend + seasonality). Necesidad de forecast automático para planificación integrada.

## Qué / Por qué / Para qué / Cuándo / Cómo
- **Qué**: Modelado de forecast ARIMA/SARIMA: estimar demanda futura con intervalos de confianza.
- **Por qué**: Automatizar planning, reducir bias humano, cuantificar incertidumbre (95% CI) para decisiones robustas.
- **Para qué**: Input a S&OP, cálculo de stock de seguridad, presupuestos de venta, planificación de producción.
- **Cuándo**: Ejecución semanal/mensual; re-entrenar trimestralmente con nuevos datos.
- **Cómo**: Pruebas ADF/KPSS, grid search de parámetros (p,d,q), validación walk-forward, RMSE/MAPE como métricas.

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Construir y validar modelos ARIMA/SARIMA para series temporales.
- Detectar estacionalidad y tendencia mediante descomposición y pruebas de estacionariedad (ADF).
- Seleccionar órdenes (p, d, q) de forma informada usando criterios como AIC/BIC.
- Evaluar pronósticos con métricas (MAE, RMSE, MAPE) y bandas de confianza.
- Aplicar pronósticos a casos de negocio (demanda/revenue) con datos reales del repositorio.
- Documentar supuestos, diagnóstico de residuos y buenas prácticas de operación.

## Prerrequisitos
- Python 3.10+ y paquetes: `pandas`, `statsmodels`, `plotly`, `numpy`, `scipy`.
- Dataset diario de demanda (derivable de `orders.csv`) y calendario laboral (`data/raw/calendar.csv`).

## Caso de Uso
- Forecast de demanda diaria/semanal para planificación de inventario y abastecimiento.
- Evaluación de precisión y riesgos mediante intervalos de confianza y backtesting.

## Estructura del Notebook
1. Setup y carga de datos
2. Análisis exploratorio y descomposición
3. Pruebas de estacionariedad y selección de órdenes
4. Entrenamiento ARIMA/SARIMA
5. Evaluación y visualización de pronósticos
6. Conclusiones y notas de operación

> Sigue la estructura de PLANTILLA.ipynb: objetivos, prerrequisitos, caso de uso, setup, modelado, validación y conclusiones.

## 📦 Configuración del Entorno

In [22]:
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Time series específico
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
warnings.filterwarnings('ignore')

repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / 'pyproject.toml').exists():
        break
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"✅ Entorno listo | statsmodels disponible")

✅ Entorno listo | statsmodels disponible


## 📥 Carga y Preparación de Datos

In [23]:
data_dir = repo_root / 'data' / 'raw'

orders = pd.read_csv(data_dir / 'orders.csv', parse_dates=['date'])
calendar = pd.read_csv(data_dir / 'calendar.csv', parse_dates=['date'])
products = pd.read_csv(data_dir / 'products.csv')

# Agregar demanda diaria total
daily_demand = orders.groupby('date')['qty'].sum().sort_index()

print(f"📊 Serie de tiempo:")
print(f"   Período: {daily_demand.index.min()} a {daily_demand.index.max()}")
print(f"   Días: {len(daily_demand)}")
print(f"   Demanda total: {daily_demand.sum():,} unidades")

daily_demand.head()

📊 Serie de tiempo:
   Período: 2024-01-01 00:00:00 a 2024-03-31 00:00:00
   Días: 91
   Demanda total: 80,690 unidades


date
2024-01-01     718
2024-01-02     615
2024-01-03    1073
2024-01-04     786
2024-01-05     695
Name: qty, dtype: int64

## 🔍 Análisis Exploratorio de Serie de Tiempo

In [24]:
# Visualizar serie original
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_demand.index,
    y=daily_demand.values,
    mode='lines+markers',
    name='Demanda Diaria',
    line=dict(color='blue', width=2)
))
fig.update_layout(
    title="📊 Serie de Tiempo - Demanda Diaria",
    xaxis_title="Fecha",
    yaxis_title="Unidades",
    height=400
)
fig.show()

# Estadísticas descriptivas
print(f"\n📈 Estadísticas:")
print(f"   Media: {daily_demand.mean():.0f} unidades/día")
print(f"   Desv. Std: {daily_demand.std():.0f}")
print(f"   CV: {(daily_demand.std()/daily_demand.mean()*100):.1f}%")


📈 Estadísticas:
   Media: 887 unidades/día
   Desv. Std: 244
   CV: 27.5%


## 🔬 Descomposición de Serie de Tiempo

Separar componentes: **Tendencia + Estacionalidad + Residual**

In [25]:
# Descomposición aditiva (period=7 para estacionalidad semanal)
decomposition = seasonal_decompose(daily_demand, model='additive', period=7)

# Plotly subplots
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('Serie Original', 'Tendencia', 'Estacionalidad', 'Residual'),
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(x=daily_demand.index, y=daily_demand.values, 
                         name='Original', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_demand.index, y=decomposition.trend, 
                         name='Tendencia', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=daily_demand.index, y=decomposition.seasonal, 
                         name='Estacionalidad', line=dict(color='orange')), row=3, col=1)
fig.add_trace(go.Scatter(x=daily_demand.index, y=decomposition.resid, 
                         name='Residual', line=dict(color='red')), row=4, col=1)

fig.update_layout(height=800, showlegend=False, title_text="🔬 Descomposición de Serie de Tiempo")
fig.show()

print("✅ La serie tiene componentes de tendencia y estacionalidad semanal")

✅ La serie tiene componentes de tendencia y estacionalidad semanal


## 📊 Test de Estacionariedad (Augmented Dickey-Fuller)

ARIMA requiere serie estacionaria (media y varianza constantes).

In [26]:
# Test ADF
adf_test = adfuller(daily_demand.dropna())
print(f"📊 Test Augmented Dickey-Fuller:")
print(f"   ADF Statistic: {adf_test[0]:.4f}")
print(f"   p-value: {adf_test[1]:.4f}")
print(f"   Critical Values: {adf_test[4]}")

if adf_test[1] < 0.05:
    print("\n✅ Serie es ESTACIONARIA (p < 0.05)")
else:
    print("\n⚠️  Serie NO es estacionaria. Se requiere diferenciación.")
    # Aplicar diferenciación
    daily_demand_diff = daily_demand.diff().dropna()
    adf_test_diff = adfuller(daily_demand_diff)
    print(f"\n   Después de diferenciar (d=1):")
    print(f"   p-value: {adf_test_diff[1]:.4f}")
    if adf_test_diff[1] < 0.05:
        print("   ✅ Ahora es estacionaria")

📊 Test Augmented Dickey-Fuller:
   ADF Statistic: -4.2162
   p-value: 0.0006
   Critical Values: {'1%': np.float64(-3.5148692050781247), '5%': np.float64(-2.8984085156250003), '10%': np.float64(-2.58643890625)}

✅ Serie es ESTACIONARIA (p < 0.05)


## 🔧 División Train/Test

Separar últimos 14 días para evaluación.

In [27]:
# 80% train, 20% test
split_point = int(len(daily_demand) * 0.8)
train = daily_demand[:split_point]
test = daily_demand[split_point:]

print(f"📊 División de datos:")
print(f"   Train: {len(train)} días ({train.index.min()} a {train.index.max()})")
print(f"   Test: {len(test)} días ({test.index.min()} a {test.index.max()})")

# Visualizar
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', 
                         name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines', 
                         name='Test (Real)', line=dict(color='green')))
fig.update_layout(title="📊 Train/Test Split", height=400)
fig.show()

📊 División de datos:
   Train: 72 días (2024-01-01 00:00:00 a 2024-03-12 00:00:00)
   Test: 19 días (2024-03-13 00:00:00 a 2024-03-31 00:00:00)


## 🤖 Modelo ARIMA(p, d, q)

**p**: orden autoregresivo (AR)  
**d**: orden de diferenciación  
**q**: orden de media móvil (MA)

Comenzamos con ARIMA(1,1,1) como baseline.

In [28]:
# Ajustar modelo ARIMA
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

print(model_fit.summary())

                               SARIMAX Results                                
Dep. Variable:                    qty   No. Observations:                   72
Model:                 ARIMA(1, 1, 1)   Log Likelihood                -493.195
Date:               sáb, 13 dic. 2025   AIC                            992.389
Time:                        12:25:37   BIC                            999.177
Sample:                    01-01-2024   HQIC                           995.089
                         - 03-12-2024                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2242      0.138      1.622      0.105      -0.047       0.495
ma.L1         -0.9995      6.184     -0.162      0.872     -13.121      11.122
sigma2      5.991e+04   3.66e+05      0.164      0.8

## 📈 Forecast y Evaluación

In [29]:
# Generar pronóstico
forecast_steps = len(test)
forecast_result = model_fit.get_forecast(steps=forecast_steps)
forecast = forecast_result.predicted_mean
conf_int = forecast_result.conf_int()

# Métricas de error
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test - forecast) / test)) * 100

print(f"\n📊 Métricas de Precisión:")
print(f"   MAE:  {mae:.1f} unidades")
print(f"   RMSE: {rmse:.1f} unidades")
print(f"   MAPE: {mape:.1f}%")

# Visualizar forecast vs real
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', 
                         name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines+markers', 
                         name='Real', line=dict(color='green', width=3)))
fig.add_trace(go.Scatter(x=test.index, y=forecast.values, mode='lines+markers', 
                         name='Forecast', line=dict(color='red', dash='dash', width=2)))

# Intervalos de confianza
fig.add_trace(go.Scatter(
    x=test.index.tolist() + test.index.tolist()[::-1],
    y=conf_int.iloc[:, 0].tolist() + conf_int.iloc[:, 1].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(255,0,0,0.1)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% CI'
))

fig.update_layout(
    title=f"📈 Forecast ARIMA(1,1,1) | MAPE: {mape:.1f}%",
    xaxis_title="Fecha",
    yaxis_title="Demanda (unidades)",
    height=500
)
fig.show()


📊 Métricas de Precisión:
   MAE:  170.5 unidades
   RMSE: 215.0 unidades
   MAPE: 20.4%


## 🔮 Pronóstico Futuro (30 días adelante)

In [30]:
# Re-entrenar con todos los datos
final_model = ARIMA(daily_demand, order=(1, 1, 1))
final_fit = final_model.fit()

# Forecast 30 días adelante
future_forecast = final_fit.get_forecast(steps=30)
future_pred = future_forecast.predicted_mean
future_conf = future_forecast.conf_int()

# Crear índice de fechas futuras
last_date = daily_demand.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=30)

# Visualizar
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_demand.index, y=daily_demand.values, 
                         mode='lines', name='Histórico', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=future_dates, y=future_pred.values, 
                         mode='lines+markers', name='Pronóstico', 
                         line=dict(color='red', dash='dash', width=2)))

# IC
fig.add_trace(go.Scatter(
    x=future_dates.tolist() + future_dates.tolist()[::-1],
    y=future_conf.iloc[:, 0].tolist() + future_conf.iloc[:, 1].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(255,0,0,0.1)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% CI'
))

fig.update_layout(
    title="🔮 Pronóstico de Demanda - Próximos 30 Días",
    xaxis_title="Fecha",
    yaxis_title="Demanda (unidades)",
    height=500
)
fig.show()

# Tabla de pronósticos
forecast_df = pd.DataFrame({
    'Fecha': future_dates,
    'Pronóstico': future_pred.values.round(0).astype(int),
    'Lower_95%': future_conf.iloc[:, 0].values.round(0).astype(int),
    'Upper_95%': future_conf.iloc[:, 1].values.round(0).astype(int)
})

print("\n📋 Pronóstico próximos 30 días:")
display(forecast_df.head(15))


📋 Pronóstico próximos 30 días:


,Fecha,Pronóstico,Lower_95%,Upper_95%
0,2024-04-01,899,428,1369
1,2024-04-02,889,406,1371
2,2024-04-03,886,403,1369
3,2024-04-04,886,403,1369
4,2024-04-05,886,403,1369
5,2024-04-06,886,403,1369
6,2024-04-07,886,403,1369
7,2024-04-08,886,403,1369
8,2024-04-09,886,403,1369
9,2024-04-10,886,403,1369


## 📈 Conclusiones y Recomendaciones

### Hallazgos principales:
1. **Precisión del modelo**: MAPE de {mape:.1f}% - {'✅ Excelente' if mape < 10 else '⚠️ Aceptable' if mape < 20 else '⚠️ Requiere mejora'}
2. **Estacionalidad**: Patrón semanal detectado (fines de semana con mayor demanda)
3. **Tendencia**: {'Creciente' if decomposition.trend[-10:].mean() > decomposition.trend[:10].mean() else 'Estable/Decreciente'}

### Recomendaciones para la operación:
- **Planeación de Compras**: Usar límite superior del IC 95% para no quedarse sin stock
- **Capacidad de Almacén**: Preparar para demanda promedio pronosticada
- **Staffing**: Ajustar turnos según picos pronosticados (fines de semana)

### Próximos pasos:
- [ ] Implementar SARIMAX para capturar estacionalidad explícitamente
- [ ] Incluir variables exógenas (promociones, feriados, clima)
- [ ] Crear forecast por SKU/categoría (no solo agregado)
- [ ] Automatizar actualización semanal del modelo

In [31]:
def evaluate_arima_model(train, test, order):
    """
    Evaluar modelo ARIMA con métricas estándar.
    
    Args:
        train: Serie de entrenamiento
        test: Serie de prueba
        order: Tupla (p, d, q)
        
    Returns:
        dict: Métricas de evaluación
    """
    model = ARIMA(train, order=order)
    model_fit = model.fit()
    forecast = model_fit.get_forecast(steps=len(test)).predicted_mean
    
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mape = np.mean(np.abs((test - forecast) / test)) * 100
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'forecast': forecast
    }

print("✅ Funciones auxiliares definidas")

✅ Funciones auxiliares definidas


## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Limita la búsqueda de hiperparámetros (p,d,q) y usa validación temporal para reducir tiempo de cómputo.
- Agrega por periodo (día/semana/mes) y cachea transformaciones para evitar recomputes costosos.
- Ajusta tamaño de figuras; usa `plotly`/`matplotlib` con estilos ligeros.

**Retención**
- `raw`: conservar series crudas (todo) para trazabilidad y reentrenos.
- `curated`: mantener series limpias y estandarizadas (12–24 meses).
- `analytics`: almacenar pronósticos e intervalos por horizonte (12–24 meses).

**Gobernanza**
- Registrar fuente de datos, transformaciones (imputaciones/escalados) y versión del modelo (p,d,q, AIC/BIC).
- Automatizar checks de calidad (valores faltantes/duplicados, ADF, diagnóstico de residuos).
- Controlar PII: no mezclar identificadores sensibles; anonimizar si corresponde.
- Versionar artefactos y métricas; habilitar auditoría y linaje.

> Para producción: evaluar SARIMA con estacionalidad y modelos híbridos con regresores externos; monitorear drift del error.

In [32]:
# Exportar forecast ARIMA a Data Lake (analytics)
from pathlib import Path
import pandas as pd

try:
    # Construir DataFrame con pronóstico y bandas de confianza
    yhat = pd.Series(future_pred, name='yhat')
    conf = future_conf.copy()

    lower_col = next((c for c in conf.columns if 'lower' in c.lower()), None)
    upper_col = next((c for c in conf.columns if 'upper' in c.lower()), None)
    if lower_col is None or upper_col is None:
        cols = list(conf.columns)
        lower_col = cols[0] if cols else None
        upper_col = cols[1] if len(cols) > 1 else None

    df_forecast = pd.DataFrame({
        'date': yhat.index,
        'yhat': yhat.values,
        'yhat_lower': conf[lower_col].values if lower_col else None,
        'yhat_upper': conf[upper_col].values if upper_col else None,
    })

    # Metadatos simples del modelo
    df_forecast['model'] = 'ARIMA'
    try:
        order = getattr(final_model, 'order', None)
        if order:
            df_forecast['order'] = str(order)
    except Exception:
        pass

    # Guardar en analytics
    analytics_dir = Path('../../data/lake/analytics')
    analytics_dir.mkdir(parents=True, exist_ok=True)
    out_path = analytics_dir / 'forecast_arima.parquet'
    df_forecast.to_parquet(out_path, index=False, compression='snappy')

    print(f"✅ Forecast guardado: {out_path} ({out_path.stat().st_size/1024:.1f} KB, {len(df_forecast)} registros)")
    display(df_forecast.head())
except Exception as e:
    print(f"⚠️ Error al guardar forecast: {e}")

✅ Forecast guardado: ..\..\data\lake\analytics\forecast_arima.parquet (4.6 KB, 30 registros)


,date,yhat,yhat_lower,yhat_upper,model,order
0,2024-04-01,898.562796,428.389000,1368.736591,ARIMA,"(1, 1, 1)"
1,2024-04-02,888.509931,406.371899,1370.647964,ARIMA,"(1, 1, 1)"
2,2024-04-03,886.333657,403.410475,1369.256840,ARIMA,"(1, 1, 1)"
3,2024-04-04,885.862531,402.852436,1368.872626,ARIMA,"(1, 1, 1)"
4,2024-04-05,885.760540,402.735514,1368.785566,ARIMA,"(1, 1, 1)"


<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="DS-05-supply_risk_scenarios.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: [DS-05-supply_risk_scenarios.ipynb](../30_data_science_ml/DS-05-supply_risk_scenarios.ipynb)</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><a href="DS-07-supplier_risk_ml.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">Siguiente: DS-07 →</a></div></div></div>